### 机器学习三大范式：监督学习、无监督学习与强化学习

> 核心认知：**监督学习**是“照猫画虎”（有标准答案）；**无监督学习**是“物以类聚”（找内在结构）；**强化学习**是“摸着石头过河”（在动态试错中求生存）。


| 维度 | **有监督学习 (Supervised)** | **无监督学习 (Unsupervised)** | **强化学习 (Reinforcement Learning)** |
| :--- | :--- | :--- | :--- |
| **数据形式** | **带标签的样本** $ (x, y) $ <br> $ y $ 是预先标注的“标准答案”。 | **纯特征** $ x $ <br> 没有任何外部标签。 | **交互轨迹** $ (S, A, R, S') $ <br> 由状态、动作、奖励、下一状态组成的动态序列。 |
| **核心目标** | **预测 / 映射** <br> 学习从 $ x $ 到 $ y $ 的映射（拟合决策边界）。 | **发现 / 描述** <br> 挖掘数据底层的隐藏结构或分布规律。 | **决策 / 控制** <br> 学习最优策略（Policy），以最大化**累积长期奖励**。 |
| **经典任务** | - 回归（房价预测）<br>- 分类（垃圾邮件识别） | - 聚类（用户分群）<br>- 降维（PCA）<br>- 关联规则（购物篮分析） | - 游戏博弈（AlphaGo）<br>- 机器人控制<br>- 自动驾驶<br>- 推荐系统长期优化 |
| **反馈机制** | **即时显式反馈（局部）** <br> 每给一个样本，损失函数（Loss）立刻算出误差。 | **无反馈（静态）** <br> 没有“对错”，只有分布合理与否（如距离度量）。 | **延迟稀疏反馈（全局）** <br> 环境只给一个分数（Reward），且可能延迟很久（**信用分配问题**）。 |
| **评估指标** | **客观绝对** <br> 准确率、精确率、召回率、RMSE（与标准答案对比）。 | **主观相对** <br> 轮廓系数、重构误差（依赖业务解释）。 | **累积目标** <br> 总回报（Return）、胜率（不关注单步对错，只看最终结果）。 |

### Q-Learning
Q-learning是一种基于价值迭代的强化学习算法，其主要特点是不需要知道环境的动态模型，智能体通过与环境的不断交互试错，最终学会在任意状态下，选择能带来最大回报的动作。

**核心思想**
维护一个被称为 Q-table 的矩阵，通过不断学习，使得Q表中每一个值都无限接近理论上的最优值，得到一张“完美”的决策表
- Q表是一个“动作价值（action-value）”备忘录,Q表的行代表智能体可能处于的所有状态（State），列代表智能体可以执行的所有动作（Action）
- Q表中的每一个数值Q(s, a)，都表示在状态s下执行动作a后，智能体期望获得的未来累计奖励的总和，Q值越大代表该动作在当前状态下越“有价值”。


**核心更新公式（贝尔曼）**

$$
Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \cdot \max_{a'} Q(s', a') - Q(s, a) \right]
$$

- $s$: 当前状态 (Current State)
- $a$: 执行的动作 (Action taken)
- $r$: 即时奖励 (Immediate reward)
- $s'$: 执行动作后智能体到达的下一个状态 (Next state)
- $\alpha$: 学习率 (Learning rate, $0 < \alpha \le 1$)，学习率越大，学习越快，但也可能更不稳定。
- $\gamma$: 折扣因子 (Discount factor, $0 \le \gamma \le 1$)，用于衡量未来奖励的重要性，越接近1越重视未来的长期收益
- $\max_{a'} Q(s', a')$: 在下一个状态 $s'$ 中，所有可能动作 $a'$ 里最大的 Q 值

**算法流程**
- 初始化：为所有状态-动作对 (s, a) 随机初始化 Q(s, a)，并将终止状态的Q值设为0。
- 对于每一个训练回合：初始化起始状态 s
- 根据当前Q值和探索策略（如ε-greedy），在状态 s 下选择一个动作 a
- 执行动作 a，观察获得的即时奖励 r 和下一个状态 s'。
- 使用贝尔曼方程更新Q表：Q(s, a) ← Q(s, a) + α [ r + γ * max<sub>a'</sub> Q(s', a') - Q(s, a) ]。
- 将当前状态更新为 s'：s ← s'。
- 如果 s' 是终止状态，则结束当前回合。
- 重复上述步骤，直到Q表收敛或达到预设的训练次数。

**平衡策略**
- 利用（Exploitation）：根据现有Q表，选择Q值最高的动作，以获取已知的最大收益。
- 探索（Exploration）：随机选择动作，即使它不是当前最优的，目的是探索未知领域，发现可能带来更大长期收益的路径。

其中，最常用的平衡策略是 ε-greedy（ε-贪心策略），在实践中ε的值通常会随着训练进程的推进而逐渐减小，即智能体在训练初期会进行大量探索，在后期则会更多地利用到已经学习到的经验。
- 以 1 - ε 的概率进行“利用”，选择当前Q值最高的动作
- 以 ε 的概率进行“探索”，在所有动作中随机选择一个。

**缺点**
- 在处理高维状态空间和连续动作空间时，Q-learning的表现不佳。
- 在大型复杂环境中，Q-learning的学习效率较低，需要大量时间和计算资源来收敛

In [4]:
import numpy as np

# 定义参数
gamma = 0.8  # 折扣因子
alpha = 0.1  # 学习率
epsilon = 0.1  # 探索概率
episodes = 10000  # 学习的总回合数

# 定义收益矩阵
R = np.array([
    [-1, -1, -1, -1, 0, -1],
    [-1, -1, -1, 0, -1, 100],
    [-1, -1, -1, 0, -1, -1],
    [-1, 0, 0, -1, 0, -1],
    [0, -1, -1, 0, -1, 100],
    [-1, 0, -1, -1, 0, 100]
])

# 初始化Q值矩阵
Q = np.zeros_like(R, dtype=float)

# Q-learning算法
for episode in range(episodes):
    # 随机选择初始状态
    state = np.random.randint(0, R.shape[0])
    
    while True:
        # 选择动作：采用ε-贪婪策略
        if np.random.rand() < epsilon:
            action = np.random.randint(0, R.shape[1])
        else:
            action = np.argmax(Q[state])

        # 执行动作，获取即时奖励和下一个状态
        next_state = action
        reward = R[state, action]
        
        # 只在奖励不为 -1 时进行 Q 值更新
        if reward != -1:
            # Q值更新
            Q[state, action] = Q[state, action] + alpha * (reward + gamma * np.max(Q[next_state]) - Q[state, action])

            # 状态转移
            state = next_state
        
        # 如果达到了终止状态，结束本回合
        if reward == 100:
            break

# 对 Q 矩阵进行规范化，使最大值为100
max_value = np.max(Q)
if max_value > 0:
    Q = (Q / max_value) * 100

# 保留两位小数
Q = np.round(Q, 2)

# 输出最终的Q值矩阵
print("最终的Q值矩阵（规范化后）：")
print(Q)

# 输出最终策略
policy = np.argmax(Q, axis=1)
print("最终策略：")
print(policy)

# 解释最终策略
for state in range(len(policy)):
    print(f"在状态 {state} 下，选择动作 {policy[state]} 作为最优策略。")


最终的Q值矩阵（规范化后）：
[[  0.     0.     0.     0.    80.     0.  ]
 [  0.     0.     0.    62.88   0.   100.  ]
 [  0.     0.     0.    64.     0.     0.  ]
 [  0.    79.9   51.18   0.    80.     0.  ]
 [ 64.     0.     0.    64.     0.   100.  ]
 [  0.    77.3    0.     0.    72.63 100.  ]]
最终策略：
[4 5 3 4 5 5]
在状态 0 下，选择动作 4 作为最优策略。
在状态 1 下，选择动作 5 作为最优策略。
在状态 2 下，选择动作 3 作为最优策略。
在状态 3 下，选择动作 4 作为最优策略。
在状态 4 下，选择动作 5 作为最优策略。
在状态 5 下，选择动作 5 作为最优策略。
